In [14]:
import pandas as pd
import numpy as np


In [15]:
df = pd.read_csv("insurance_claim_fraud_dataset.csv")

print("Original shape:", df.shape)


Original shape: (100000, 62)


In [16]:
date_cols = [
    "claim_date",
    "incident_date",
    "policy_start_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Reporting delay
df["reporting_delay_days"] = (
    df["claim_date"] - df["incident_date"]
).dt.days

# Policy age
df["policy_age_days"] = (
    df["claim_date"] - df["policy_start_date"]
).dt.days


In [17]:

df["claim_to_coverage_ratio"] = (
    df["claim_amount"] /
    df["coverage_amount"].replace(0, np.nan)
)

df["claim_to_income_ratio"] = (
    df["claim_amount"] /
    df["annual_income"].replace(0, np.nan)
)

df["vehicle_age"] = (
    df["claim_date"].dt.year - df["vehicle_year"]
)



In [18]:
binary_cols = [
    "suspicious_documents",
    "delayed_reporting",
    "inconsistent_statements",
    "claim_submitted_at_night",
    "multiple_claims_last_year"
]

for col in binary_cols:

    if col in df.columns:

        df[col + "_binary"] = (
            df[col]
            .astype(str)
            .str.lower()
            .str.strip()
            .map({
                "yes": 1,
                "no": 0
            })
            .fillna(0)
        )

In [19]:
df["fraud_indicator_count"] = (
    df["suspicious_documents_binary"] +
    df["delayed_reporting_binary"] +
    df["inconsistent_statements_binary"] +
    df["claim_submitted_at_night_binary"] +
    df["multiple_claims_last_year_binary"]
)


# ---------------------------------------------------------
# 5. HIGH-RISK COMBINATION
# ---------------------------------------------------------

df["high_risk_combination"] = (
    (df["suspicious_documents_binary"] == 1) &
    (df["delayed_reporting_binary"] == 1) &
    (df["inconsistent_statements_binary"] == 1)
).astype(int)


In [20]:

columns_to_remove = [

    # IDs / personal information
    "claim_id",
    "policy_id",
    "customer_id",
    "customer_name",

    # Original dates
    "claim_date",
    "incident_date",
    "policy_start_date",

    # Vehicle identification
    "vehicle_make",
    "vehicle_model",

    # Post-claim / unnecessary fields
    "approved_amount",
    "payout_amount",
    "claim_status",
    "repair_cost",
    "medical_cost",
    "towing_cost",
    "rental_vehicle_cost",
    "total_loss"
]

df.drop(
    columns=columns_to_remove,
    errors="ignore",
    inplace=True
)



In [21]:

original_binary_cols = [
    "suspicious_documents",
    "delayed_reporting",
    "inconsistent_statements",
    "claim_submitted_at_night",
    "multiple_claims_last_year"
]

df.drop(
    columns=original_binary_cols,
    errors="ignore",
    inplace=True
)

In [22]:

numeric_cols = df.select_dtypes(
    include=["int64", "float64"]
).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())


categorical_cols = df.select_dtypes(
    include=["object"]
).columns

for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")


In [24]:

print("Reduced columns:", len(df.columns))

print("\nRemaining columns:")
for col in df.columns:
    print(col)

print("\nReduced dataset shape:")
print(df.shape)


Reduced columns: 51

Remaining columns:
claim_amount
deductible
claim_duration_days
age
gender
marital_status
occupation
annual_income
education_level
credit_score
years_with_company
previous_claims
policy_type
policy_duration_years
premium_amount
coverage_amount
deductible_plan
payment_method
policy_active
incident_type
accident_severity
incident_location
weather_condition
police_report
witnesses
injuries
hospital_visit
property_damage
vehicle_damage
vehicle_year
vehicle_age
mileage
vehicle_value
ownership_type
gps_location_match
repair_shop_verified
identity_verified
claim_history_risk
fraud_risk_score
fraudulent
reporting_delay_days
policy_age_days
claim_to_coverage_ratio
claim_to_income_ratio
suspicious_documents_binary
delayed_reporting_binary
inconsistent_statements_binary
claim_submitted_at_night_binary
multiple_claims_last_year_binary
fraud_indicator_count
high_risk_combination

Reduced dataset shape:
(100000, 51)


In [25]:
df.to_csv(
    "insurance_claim_fraud_reduced.csv",
    index=False
)

print("\nSaved as: insurance_claim_fraud_reduced.csv")


Saved as: insurance_claim_fraud_reduced.csv
